In [ ]:
import pandas as pd

In [ ]:
latest_df = pd.read_csv(
    "../../pipeline/pipeline_steps/input_files/2012-01-2024-08-overdoses.csv"
)

In [ ]:
latest_df = latest_df.drop(columns=["DateofBirth"])

In [ ]:
latest_df["DeathDate"] = pd.to_datetime(latest_df["DeathDate"])

In [ ]:
latest_df["MonthYear"] = latest_df["DeathDate"].apply(lambda x: x.strftime("%Y-%m"))
latest_df["Year"] = latest_df["DeathDate"].apply(lambda x: x.year)
latest_df["YearWeek"] = latest_df["DeathDate"].apply(lambda x: x.strftime("%Y-%U"))

In [ ]:
latest_df_w_zips = latest_df.dropna(subset="ZIPCODE")

In [ ]:
latest_df_w_zips["ZIPCODE"] = latest_df_w_zips["ZIPCODE"].astype(int).astype(str)

In [ ]:
# Define bins and labels
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
labels = [
    "0-9",
    "10-19",
    "20-29",
    "30-39",
    "40-49",
    "50-59",
    "60-69",
    "70-79",
    "80-89",
    "90+",
]

# Assign each age to a bin
latest_df_w_zips["Age_Bin"] = pd.cut(
    latest_df_w_zips["Age"], bins=bins, labels=labels, right=False
)

In [ ]:
def reassign_gender(row):
    if row == "M":
        return "male"
    elif row == "F":
        return "female"
    else:
        return row.casefold()

In [ ]:
latest_df_w_zips["Gender"] = latest_df_w_zips["Gender"].apply(reassign_gender)

In [ ]:
# Standardize gender values
latest_df_w_zips["Gender"] = (
    latest_df_w_zips["Gender"].str.strip().str.lower()
)  # Normalize case & trim spaces

# Define standardization mapping
gender_mapping = {
    "male": "Male",
    "female": "Female",
    "m": "Male",
    "f": "Female",
    "MALE": "Male",
    "FEMALE": "Female",
}

# Apply mapping & replace invalid values with "Unknown"
latest_df_w_zips["Gender"] = latest_df_w_zips["Gender"].replace(gender_mapping)

# Handle missing or empty values
latest_df_w_zips["Gender"] = latest_df_w_zips["Gender"].replace(
    {"": "Unknown"}
)  # Replace empty strings
latest_df_w_zips["Gender"] = latest_df_w_zips["Gender"].fillna(
    "Unknown"
)  # Replace NaNs

In [ ]:
latest_df_w_zips["Race"] = latest_df_w_zips["Race"].fillna("UNKNOWN")

In [ ]:
latest_df_w_zips = latest_df_w_zips.dropna(subset="CT20")

In [ ]:
latest_df_w_zips["CT20"] = latest_df_w_zips["CT20"].astype(int).astype(str)

In [ ]:
latest_df_w_zips.to_csv(
    "../../reports/deidentified_overdose_201201202408_zips_0311.csv"
)

In [ ]:
latest_df.to_csv("../../reports/deidentified_overdose_201201202408.csv")